In [16]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Mac-safe non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter
import warnings
import joblib

In [ ]:
df = pd.read_csv('bs140513_032310.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())


Shape: (594643, 10)

Columns: ['step', 'customer', 'age', 'gender', 'zipcodeOri', 'merchant', 'zipMerchant', 'category', 'amount', 'fraud']

Dtypes:
 step             int64
customer           str
age                str
gender             str
zipcodeOri         str
merchant           str
zipMerchant        str
category           str
amount         float64
fraud            int64
dtype: object

First 5 rows:
    step       customer  age gender zipcodeOri       merchant zipMerchant  \
0     0  'C1093826151'  '4'    'M'    '28007'   'M348934600'     '28007'   
1     0   'C352968107'  '2'    'M'    '28007'   'M348934600'     '28007'   
2     0  'C2054744914'  '4'    'F'    '28007'  'M1823072687'     '28007'   
3     0  'C1760612790'  '3'    'M'    '28007'   'M348934600'     '28007'   
4     0   'C757503768'  '5'    'M'    '28007'   'M348934600'     '28007'   

              category  amount  fraud  
0  'es_transportation'    4.55      0  
1  'es_transportation'   39.68      0  
2  'es_transp

In [ ]:
print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Statistical Summary ---")
print(df.describe())

print("\n--- Class Distribution ---")
fraud_counts = df['fraud'].value_counts()
fraud_pct = df['fraud'].value_counts(normalize=True) * 100
print(fraud_counts)
print(fraud_pct.round(2))


--- Missing Values ---
step           0
customer       0
age            0
gender         0
zipcodeOri     0
merchant       0
zipMerchant    0
category       0
amount         0
fraud          0
dtype: int64

--- Statistical Summary ---
                step         amount          fraud
count  594643.000000  594643.000000  594643.000000
mean       94.986827      37.890135       0.012108
std        51.053632     111.402831       0.109369
min         0.000000       0.000000       0.000000
25%        52.000000      13.740000       0.000000
50%        97.000000      26.900000       0.000000
75%       139.000000      42.540000       0.000000
max       179.000000    8329.960000       1.000000

--- Class Distribution ---
fraud
0    587443
1      7200
Name: count, dtype: int64
fraud
0    98.79
1     1.21
Name: proportion, dtype: float64


In [9]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('BankSim EDA — Credit Card Fraud Detection', fontsize=16)

# 3a. Class imbalance bar chart
axes[0, 0].bar(['Legitimate (0)', 'Fraud (1)'], fraud_counts.values,
               color=['steelblue', 'tomato'], edgecolor='black')
axes[0, 0].set_title('Class Distribution')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0, 0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# 3b. Transaction amount distribution
axes[0, 1].hist(df[df['fraud'] == 0]['amount'], bins=60, alpha=0.6,
                color='steelblue', label='Legitimate', density=True)
axes[0, 1].hist(df[df['fraud'] == 1]['amount'], bins=60, alpha=0.6,
                color='tomato', label='Fraud', density=True)
axes[0, 1].set_title('Amount Distribution by Class')
axes[0, 1].set_xlabel('Transaction Amount')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# 3c

if 'category' in df.columns:
    cat_fraud = df.groupby('category')['fraud'].mean().sort_values(ascending=False)
    cat_fraud.plot(kind='bar', ax=axes[0, 2], color='coral', edgecolor='black')
    axes[0, 2].set_title('Fraud Rate by Merchant Category')
    axes[0, 2].set_ylabel('Fraud Rate')
    axes[0, 2].tick_params(axis='x', rotation=45)

# 3d. Transaction count per step (time proxy)
if 'step' in df.columns:
    step_counts = df.groupby('step')['fraud'].value_counts().unstack(fill_value=0)
    step_counts.plot(ax=axes[1, 0], color=['steelblue', 'tomato'], linewidth=0.8)
    axes[1, 0].set_title('Transactions Over Time (Steps)')
    axes[1, 0].set_xlabel('Step')
    axes[1, 0].set_ylabel('Count')

# 3e. Boxplot — amount by fraud
df.boxplot(column='amount', by='fraud', ax=axes[1, 1],
           notch=False, patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
axes[1, 1].set_title('Amount by Fraud Class (Boxplot)')
axes[1, 1].set_xlabel('Fraud')
axes[1, 1].set_ylabel('Amount')
plt.sca(axes[1, 1])
plt.title('Amount by Fraud Class')

# 3f. Correlation heatmap (numeric only)
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, ax=axes[1, 2], annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, square=True)
axes[1, 2].set_title('Correlation Heatmap')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print("EDA plot saved: eda_overview.png")


EDA plot saved: eda_overview.png


In [10]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('eda_plots', exist_ok=True)

# ─── 1. Class Imbalance Bar Chart ─────────────────────────────
fraud_counts = df['fraud'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(['Legitimate (0)', 'Fraud (1)'], fraud_counts.values,
       color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title('Class Distribution', fontsize=14)
ax.set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_plots/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 01_class_distribution.png")

# ─── 2. Transaction Amount Distribution by Class ──────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df[df['fraud'] == 0]['amount'], bins=60, alpha=0.6,
        color='steelblue', label='Legitimate', density=True)
ax.hist(df[df['fraud'] == 1]['amount'], bins=60, alpha=0.6,
        color='tomato', label='Fraud', density=True)
ax.set_title('Transaction Amount Distribution by Class', fontsize=14)
ax.set_xlabel('Transaction Amount')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('eda_plots/02_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 02_amount_distribution.png")

# ─── 3. Fraud Rate by Merchant Category ───────────────────────
if 'category' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    cat_fraud = df.groupby('category')['fraud'].mean().sort_values(ascending=False)
    cat_fraud.plot(kind='bar', ax=ax, color='coral', edgecolor='black')
    ax.set_title('Fraud Rate by Merchant Category', fontsize=14)
    ax.set_ylabel('Fraud Rate')
    ax.set_xlabel('Category')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('eda_plots/03_fraud_rate_by_category.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 03_fraud_rate_by_category.png")

# ─── 4. Transaction Count Over Time (Steps) ──────────────────
if 'step' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    step_counts = df.groupby('step')['fraud'].value_counts().unstack(fill_value=0)
    step_counts.columns = ['Legitimate', 'Fraud']
    step_counts.plot(ax=ax, color=['steelblue', 'tomato'], linewidth=0.9)
    ax.set_title('Transaction Volume Over Time (Steps)', fontsize=14)
    ax.set_xlabel('Step')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.savefig('eda_plots/04_transactions_over_time.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 04_transactions_over_time.png")

# ─── 5. Boxplot — Amount by Fraud Class ───────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fraud_groups = [df[df['fraud'] == 0]['amount'].values,
                df[df['fraud'] == 1]['amount'].values]
bp = ax.boxplot(fraud_groups, patch_artist=True, notch=False,
                labels=['Legitimate (0)', 'Fraud (1)'])
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('tomato')
ax.set_title('Transaction Amount by Fraud Class', fontsize=14)
ax.set_ylabel('Amount')
ax.set_xlabel('Class')
plt.tight_layout()
plt.savefig('eda_plots/05_amount_boxplot.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 05_amount_boxplot.png")

# ─── 6. Correlation Heatmap ────────────────────────────────────
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, square=True)
ax.set_title('Correlation Heatmap (Numeric Features)', fontsize=14)
plt.tight_layout()
plt.savefig('eda_plots/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 06_correlation_heatmap.png")

# ─── 7. Fraud Count by Customer (Top 20) ──────────────────────
if 'customer' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    top_custs = df[df['fraud'] == 1]['customer'].value_counts().head(20)
    top_custs.plot(kind='bar', ax=ax, color='tomato', edgecolor='black')
    ax.set_title('Top 20 Customers by Fraud Transaction Count', fontsize=14)
    ax.set_xlabel('Customer ID')
    ax.set_ylabel('Fraud Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('eda_plots/07_top_fraud_customers.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 07_top_fraud_customers.png")

# ─── 8. Fraud Amount vs Legitimate Amount — Violin Plot ───────
fig, ax = plt.subplots(figsize=(8, 6))
plot_data = pd.DataFrame({
    'amount': df['amount'],
    'class': df['fraud'].map({0: 'Legitimate', 1: 'Fraud'})
})
sns.violinplot(data=plot_data, x='class', y='amount',
               palette={'Legitimate': 'steelblue', 'Fraud': 'tomato'},
               ax=ax, inner='quartile')
ax.set_title('Amount Distribution — Violin Plot', fontsize=14)
ax.set_xlabel('Class')
ax.set_ylabel('Transaction Amount')
plt.tight_layout()
plt.savefig('eda_plots/08_amount_violin.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 08_amount_violin.png")

# ─── 9. Missing Values Heatmap ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, ax=ax,
            yticklabels=False, cmap='viridis')
ax.set_title('Missing Values Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('eda_plots/09_missing_values.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: 09_missing_values.png")

# ─── 10. Class-wise Fraud Rate per Step (Heatmap) ────────────
if 'step' in df.columns and 'category' in df.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    pivot = df.groupby(['step', 'category'])['fraud'].mean().unstack(fill_value=0)
    sns.heatmap(pivot.T, ax=ax, cmap='YlOrRd', linewidths=0.3)
    ax.set_title('Fraud Rate per Step × Category', fontsize=14)
    ax.set_xlabel('Step')
    ax.set_ylabel('Category')
    plt.tight_layout()
    plt.savefig('eda_plots/10_fraud_rate_step_category.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Saved: 10_fraud_rate_step_category.png")

print("\n✅ All EDA plots saved to 'eda_plots/' folder.")

Saved: 01_class_distribution.png
Saved: 02_amount_distribution.png
Saved: 03_fraud_rate_by_category.png
Saved: 04_transactions_over_time.png
Saved: 05_amount_boxplot.png
Saved: 06_correlation_heatmap.png


/var/folders/x1/45thx8vn3tj4rd1bg_pxj_y00000gn/T/ipykernel_10857/1444924514.py:73: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(fraud_groups, patch_artist=True, notch=False,


Saved: 07_top_fraud_customers.png


/var/folders/x1/45thx8vn3tj4rd1bg_pxj_y00000gn/T/ipykernel_10857/1444924514.py:117: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=plot_data, x='class', y='amount',


Saved: 08_amount_violin.png
Saved: 09_missing_values.png
Saved: 10_fraud_rate_step_category.png

✅ All EDA plots saved to 'eda_plots/' folder.


In [12]:
df_proc = df.copy()

# ─── 4a. Drop constant/high-cardinality columns ───────────────
drop_cols = ['zipcodeOri', 'zipMerchant']
df_proc.drop(columns=[c for c in drop_cols if c in df_proc.columns], inplace=True)

# ─── 4b. Aggregate BEFORE label encoding ──────────────────────
# Must use df_proc (with string customer/merchant) BEFORE encoding
if 'customer' in df_proc.columns:
    cust_stats = df_proc.groupby('customer')['amount'].agg(
        cust_txn_count='count',
        cust_avg_amount='mean',
        cust_std_amount='std',
        cust_max_amount='max'
    ).reset_index()
    df_proc = df_proc.merge(cust_stats, on='customer', how='left')
    print("Customer aggregate features added.")

if 'merchant' in df_proc.columns:
    merch_stats = df_proc.groupby('merchant')['amount'].agg(
        merch_txn_count='count',
        merch_avg_amount='mean',
    ).reset_index()
    # Fraud rate per merchant (safe: uses df_proc which still has 'fraud')
    merch_fraud = df_proc.groupby('merchant')['fraud'].mean().reset_index()
    merch_fraud.rename(columns={'fraud': 'merch_fraud_rate'}, inplace=True)
    merch_stats = merch_stats.merge(merch_fraud, on='merchant', how='left')
    df_proc = df_proc.merge(merch_stats, on='merchant', how='left')
    print("Merchant aggregate features added.")

# ─── 4c. Label encode categorical columns AFTER aggregation ───
cat_cols = df_proc.select_dtypes(include=['object', 'string']).columns.tolist()
le_dict = {}  # store encoders for inverse transform later if needed
for col in cat_cols:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))
    le_dict[col] = le
    print(f"Label encoded: {col}")

# ─── 4d. Fill NaN (e.g. std of single-transaction customers) ──
df_proc.fillna(0, inplace=True)

print("\nProcessed DataFrame shape:", df_proc.shape)
print("Columns after engineering:", df_proc.columns.tolist())

Customer aggregate features added.
Merchant aggregate features added.
Label encoded: customer
Label encoded: age
Label encoded: gender
Label encoded: merchant
Label encoded: category

Processed DataFrame shape: (594643, 15)
Columns after engineering: ['step', 'customer', 'age', 'gender', 'merchant', 'category', 'amount', 'fraud', 'cust_txn_count', 'cust_avg_amount', 'cust_std_amount', 'cust_max_amount', 'merch_txn_count', 'merch_avg_amount', 'merch_fraud_rate']


In [13]:
if 'step' in df_proc.columns:
    df_proc_sorted = df_proc.sort_values('step').reset_index(drop=True)
    split_idx = int(len(df_proc_sorted) * 0.8)
    train_df  = df_proc_sorted.iloc[:split_idx]
    test_df   = df_proc_sorted.iloc[split_idx:]
    print(f"\nTemporal split — Train: {len(train_df):,} | Test: {len(test_df):,}")
else:
    train_df, test_df = train_test_split(df_proc, test_size=0.2,
                                          stratify=df_proc['fraud'], random_state=42)

X_train = train_df.drop(columns=['fraud'])
y_train = train_df['fraud']
X_test  = test_df.drop(columns=['fraud'])
y_test  = test_df['fraud']



Temporal split — Train: 475,714 | Test: 118,929


In [17]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved.")


Scaler saved.


In [19]:
from sklearn.neighbors import NearestNeighbors

# Pass n_jobs via the NearestNeighbors estimator — works in all modern imblearn versions
knn = NearestNeighbors(n_neighbors=5, n_jobs=-1)
smote = SMOTE(random_state=42, k_neighbors=knn)

print(f"\nBefore SMOTE: {Counter(y_train)}")
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"After SMOTE:  {Counter(y_train_smote)}")


Before SMOTE: Counter({0: 469794, 1: 5920})
After SMOTE:  Counter({0: 469794, 1: 469794})


In [20]:
train_out = X_train_scaled.copy(); train_out['fraud'] = y_train.values
test_out  = X_test_scaled.copy();  test_out['fraud']  = y_test.values
smote_out = pd.DataFrame(X_train_smote, columns=X_train.columns)
smote_out['fraud'] = y_train_smote

train_out.to_csv('train_processed.csv', index=False)
test_out.to_csv('test_processed.csv',   index=False)
smote_out.to_csv('train_smote.csv',     index=False)
print("\n✅ Saved: train_processed.csv, test_processed.csv, train_smote.csv")



✅ Saved: train_processed.csv, test_processed.csv, train_smote.csv
